In [ ]:
import logging
import nest_asyncio
import sys
from llama_index.core import Settings
from llama_index.core.llama_dataset.generator import RagDatasetGenerator
from llama_index.llms.litellm import LiteLLM

from src.food_rag.utils import hummus_docify
from src.persona import persona_generator
from src.persona import qa_generator
from src.persona.qa_generator import QuestionType
from src.utils import load_util
from src.utils import rdf_util

logging.basicConfig(
    stream=sys.stdout, level=logging.WARN
)  # logging.INFO or even logging.DEBUG for more verbose output
# os.environ['LITELLM_LOG'] = 'INFO'

nest_asyncio.apply()  # Required to run the notebook in Jupyter for disabling async operations in the notebook

In [ ]:
# Output Path
data_path = './data'
qa_name = 'small_1'

# Load configs
nutrition_config = load_util.get_config('config/nutrition_config.yml')
persona_config = load_util.get_config('config/persona_config.yml')

# Load key
api_key = load_util.load_file('config/api_key')

# 1. Setup

## 1.1 Load Preprocessed Data

In [ ]:
# Load the current recipes data
hummus_data = load_util.get_current_hummus(data_path)
food_on = rdf_util.PreProcessedFoodOn(hummus_data).preprocess()

## 1.2 Init LLM

Alternatively a local LLM: [`llama-2-chat-13b-ggml`](https://huggingface.co/TheBloke/Llama-2-13B-chat-GGML) model.

Full setup instructions/details at [Llama CPP guide](https://gpt-index.readthedocs.io/en/stable/examples/llm/llama_2_llama_cpp.html).

In [ ]:
llm = LiteLLM(  # Doc see @ --> https://docs.litellm.ai/docs/providers/openai_compatible
    api_base="https://....",
    api_key=api_key,
    model="openai/llama3.1"
)

# Set Global Settings
Settings.llm = llm

# 2. Generate Questions
## 2.0 Investigate Persona and Meal Plan

In [ ]:
# Generate example persona
persona_requirements = persona_generator.generate_persona_preferences(persona_config, nutrition_config, hummus_data, food_on=food_on, print_persona=True)

In [ ]:
# Generate example meal plan
meal_plan = persona_generator.get_random_meal_plan_nutrients(persona_config)
meal_plan

## 2.1 Our Questions


In [ ]:
# Generate QA set for recipes
recipe_question_answer_set = qa_generator.generate_question_answer_set(hummus_data, QuestionType.RECIPE, persona_config, nutrition_config, data_path=data_path, sub_dir_name=qa_name, llm=llm, size=10, batch_size=10, storage_frequency=10, food_on=food_on)

In [ ]:
# Generate questions for the meal plan
meal_plan_question_answer_set = qa_generator.generate_question_answer_set(hummus_data, QuestionType.MEAL_PLAN, persona_config, nutrition_config, data_path=data_path, sub_dir_name=qa_name, llm=llm, size=10,batch_size=10, storage_frequency=10, food_on=food_on)

In [ ]:
# Remove duplicated questions (solely dependent on persona and meal plan requirements) # Note: Not saved on disk
recipe_question_answer_set = qa_generator.remove_duplicates(recipe_question_answer_set)
meal_plan_question_answer_set = qa_generator.remove_duplicates(meal_plan_question_answer_set)

In [ ]:
# Load, automatically adds full recipes instead of just the list of urls 
question_answer_sets = qa_generator.load_qa_set(data_path, sub_dir_name=qa_name)

## 2.2 LLM Generated
Instead of generating questions based on our persona (and then using an LLM to reformulate), we can directly use the LLM to generate questions. However, questions generated by the LLM might not be as relevant as the ones generated by our persona.
Refer to https://docs.llamaindex.ai/en/stable/module_guides/evaluating/usage_pattern/

TODO: Just prototype, not in use right now.

In [ ]:
# Convert the recipes data to documents
documents = hummus_docify.recipes_to_docs(hummus_data['recipes_df'])

# Just select the first 10 documents for now
documents = documents[:10]

In [ ]:
# define generator
# Todo the stored input documents seem to be limited by size --> investigate!
dataset_generator = RagDatasetGenerator.from_documents(
    documents=documents,
    llm=llm,
    num_questions_per_chunk=10,  # set the number of questions per nodes
    show_progress=True,
)

In [ ]:
# generate questions
# Note: For 10*10 questions it took ~8 minutes.
rag_dataset = dataset_generator.generate_questions_from_nodes()
questions = [e.query for e in rag_dataset.examples]

In [ ]:
# Store the questions in a file
load_util.ensure_dir(data_path + '/qa')
rag_dataset.save_json(data_path + "/qa/rag_dataset.json")

In [ ]:
# Load the questions from the file
# rag_dataset = load_util.load_file(data_path + "/qa/rag_dataset.json")
# questions = [e['query'] for e in rag_dataset['examples']]